In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.script.variable_profiling import eda_per_table_printing_results
from default_risk.script.variable_profiling import eda_per_table_persisting_result_html
from default_risk.script.variable_profiling import create_files_nulls_per_colmun
import default_risk.config as cfg
import logging
import dtale
import gc



credit_card_df= pd.read_csv(cfg.CREDIT_CARD_BALANCE)

credit_card_df.sort_values(["SK_ID_PREV","MONTHS_BALANCE"],inplace=True)


log = logging.getLogger('werkzeug')

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)


#auxiliar functions

def recreate_and_sort_the_serie_given_ids(ids : pd.Series) -> pd.DataFrame :
    recreated_series= credit_card_df[credit_card_df["SK_ID_PREV"].isin(ids)]
    return recreated_series.sort_values(["SK_ID_PREV","MONTHS_BALANCE"])
     

def recreate_and_sort_series_given_rows(rows : pd.DataFrame) -> pd.DataFrame :
    id_rows= rows["SK_ID_PREV"].unique()
    return recreate_and_sort_the_serie_given_ids(id_rows)



Invariants founded at the moment:

1- CNT_INSTALMENT_MATURE_CUM is monotonically non-decreasing over the time (MONTHS_BALANCE). Start to increment when AMT_INST_MIN_REGULARITY become < 0 and stop when AMT_BALANCE = 0
or in a more formal definition, if Xt-1[CNT_INSTALMENT_MATURE_CUM] > 0 (we are not at the beggining of the temporal serie) and Xt-1[CNT_INSTALMENT_MATURE_CUM] == Xt[CNT_INSTALMENT_MATURE_CUM] then Xt-1[AMT_BALANCE] == 0  (100%)

1- AMT_BALANCE > AMT_RECEIVABLE_PRINCIPAL (100%)

2- AMT_TOTAL_RECEIVABLE >= AMT_RECIVABLE  (100%)

3- AMT_TOTAL_RECEIVABLE >= AMT_RECEIVABLE_PRINCIPAL (97,21%)

4- AMT_BALANCE & AMT_RECEIVABLE_PRINCIPAL > 0 (99.04%)




In [ ]:
#files for the data dictionary
create_files_nulls_per_colmun(credit_card_df,"credit_card_balance")

In [ ]:
#run the screening script on bureau
eda_per_table_printing_results(credit_card_df, schema, "credit_card_balance",False)

In [ ]:
#In order to compare magnitudes, his semantic meanings and how reliavable are in this dataset we proceed to this validations:

data_frame_size=len(credit_card_df)

recivable_bigger_than_the_total= len(credit_card_df[credit_card_df["AMT_RECIVABLE"] > credit_card_df["AMT_TOTAL_RECEIVABLE"]])
print(str(recivable_bigger_than_the_total) + " of cases where AMT_RECIVABLE are bigger than AMT_TOTAL_RECEIVABLE ")


capital_bigger_than_total= (credit_card_df["AMT_RECEIVABLE_PRINCIPAL"] > credit_card_df["AMT_TOTAL_RECEIVABLE"]) & ( credit_card_df["CNT_INSTALMENT_MATURE_CUM"] > 1)
porcentaje_of_capital_invariance_violation= (len(credit_card_df[capital_bigger_than_total]) * 100) / data_frame_size
print(str(len(credit_card_df[capital_bigger_than_total])) + " of cases where AMT_RECEIVABLE_PRINCIPAL are bigger than AMT_TOTAL_RECEIVABLE (once when the payment is ongoing)")
print("that represent a " + str(porcentaje_of_capital_invariance_violation) + "%" + " of cases with violation of this invariant")

capital_bigger_than_balance= len(credit_card_df[credit_card_df["AMT_RECEIVABLE_PRINCIPAL"] > credit_card_df["AMT_BALANCE"]])
print(str(capital_bigger_than_balance) + " of cases where AMT_RECEIVABLE_PRINCIPAL is bigger than AMT_BALANCE")


negative_balance= (credit_card_df["AMT_BALANCE"] < 0).sum()
print(str(negative_balance) + " of cases where AMT_BALANCE is negative")
positive_balance_invariant_porcentaje= ((credit_card_df["AMT_BALANCE"] < 0).sum() * 100) / data_frame_size
print("that represent a " + str(positive_balance_invariant_porcentaje) + "%" + " of cases with violation of this invariant")

0 of cases where AMT_RECIVABLE are bigger than AMT_TOTAL_RECEIVABLE 
107270 of cases where AMT_RECEIVABLE_PRINCIPAL are bigger than AMT_TOTAL_RECEIVABLE (once when the payment is ongoing)
that represent a 2.7932626307445854% of cases with violation of this invariant
0 of cases where AMT_RECEIVABLE_PRINCIPAL is bigger than AMT_BALANCE
2345 of cases where AMT_BALANCE is negative
that represent a 0.06106274698514079% of cases with violation of this invariant


In [ ]:
#we identify a high degree of collinearity between AMT_BALANCE and AMT_TOTAL_RECEIVABLE. 
diff = credit_card_df["AMT_BALANCE"] - credit_card_df["AMT_TOTAL_RECEIVABLE"]
non_cero_ammount= (diff != 0).sum()
print("this 2 columns are different in: " + str((non_cero_ammount * 100) / len(diff)) + "% the observations")
diff.describe()
#With 84% of the observations having this 2 columns with equal values but the difference carrying potential signal we decides to:
#1- Drop AMT_TOTAL_RECIVABLE to avoid redundance
#2- Create a feature of the difference
#3- Apply Symlog transformation to the new feature based on the metrics of the .describe() exhibit a distribution dominated by outliers.

this 2 columns are different in: 15.500615574984533% the observations


count    3.840312e+06
mean     2.018698e+02
std      1.558639e+03
min     -1.228500e+05
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.983500e+05
dtype: float64

In [13]:
#In the same line, we explore the colineality between "AMT_PAYMENT_CURRENT" & AMT_PAYMENT_TOTAL_CURRENT"
diff = credit_card_df["AMT_PAYMENT_CURRENT"] - credit_card_df["AMT_PAYMENT_TOTAL_CURRENT"]
non_cero_ammount= (diff != 0).sum()
print("this 2 columns are different in: " + str((non_cero_ammount * 100) / len(diff)) + "% the observations")
diff.describe()
#Thes are not redundant but are mathematically linked. Extra_payed_costs = AMT_PAYMENT_TOTAL_CURRENT - AMT_PAYMENT_CURRENT.
#Therefore in order to explicit relationships hard to discover for the model and avoid colineality we will:
#1- Calculate the diference and save in a new column
#2- Drop AMT_PAYMENT_CURRENT because we consider it less reliable and have (19% of missing values) and keep AMT_PAYMENT_TOTAL_CURRENT as anchor (has no missing values)

this 2 columns are different in: 50.913050814621315% the observations


count    3.072324e+06
mean     7.946965e+02
std      3.704636e+03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.173500e+02
max      1.802022e+06
dtype: float64

In [40]:
first_ids_prev_app= (credit_card_df["SK_ID_PREV"].unique())[:1000]
dtale.show(recreate_and_sort_the_serie_given_ids(first_ids_prev_app))

2026-04-24 00:21:21,031 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\defa

In [ ]:
#This proof the existence of rows where the minimun ammout to pay that month to avoid delinquency is missing but exist a debt.
rows_with_null= credit_card_df[credit_card_df["AMT_INST_MIN_REGULARITY"].isna()]
mask_exist_debt= rows_with_null["AMT_BALANCE"].notnull() & rows_with_null["AMT_BALANCE"] != 0
rows_with_null_and_debt= rows_with_null[mask_exist_debt]
dtale.show(recreate_and_sort_series_given_rows(rows_with_null_and_debt))
#this pattern tends to show in the first moment the serie represent a debt, and no necessarily are cases of delinquency.

In [ ]:
rows_with_null= credit_card_df[credit_card_df["AMT_INST_MIN_REGULARITY"].isna()]
mask_exist_deb_and_payed_instalments= (rows_with_null["AMT_BALANCE"].notnull()) & (rows_with_null["AMT_BALANCE"] != 0) & (rows_with_null["CNT_INSTALMENT_MATURE_CUM"] > 0)
rows_with_null_and_debt= rows_with_null[mask_exist_deb_and_payed_instalments]
recreate_and_sort_series_given_rows(rows_with_null_and_debt).head()
#this represent that once the client start to repair the debt, always have definied the minimun ammout to avoid delincuency defined 

In [ ]:
#in order to check for the behaivor of nulls and if are legit missing values or corrupted data, we let's continue with "CNT_INSTALMENT_MATURE_CUM".
rows_with_null_values = credit_card_df[credit_card_df["CNT_INSTALMENT_MATURE_CUM"].isna()]
dtale.show(recreate_and_sort_series_given_rows(rows_with_null_values))
#this give us the strong hipotesis that "CNT_INSTALMENT_MATURE_CUM" missing values are "silents ceros".

In [ ]:
#In order to test the hipotesis of "CNT_INSTALMENT_MATURE_CUM" nulls being "silents ceros"
def is_valid_next(series):
    return series.isin([0, 1]) | series.isna()

credit_card_df["NEXT_VALUE_INSTALMENT"] = credit_card_df.groupby("SK_ID_PREV")["CNT_INSTALMENT_MATURE_CUM"].shift(-1)
mask_for_nulls_pattern= (credit_card_df["CNT_INSTALMENT_MATURE_CUM"].isnull()) & ~(is_valid_next(credit_card_df["NEXT_VALUE_INSTALMENT"]))
recreate_and_sort_series_given_rows(credit_card_df[mask_for_nulls_pattern]).head()
#this proof the hipotesis that "CNT_INSTALMENT_MATURE_CUM" missing values are just "silents ceros"


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF,NEXT_VALUE_INSTALMENT


In [7]:
#We discover a relatioship between "CNT_INSTALMENT_MATURE_CUM" and "AMT_BALANCE", now we want to visualize if there is exception to this.
credit_card_df["CNT_INSTALMENT_IS_CONSTANT_VS_PREV"] = (
    credit_card_df.groupby("SK_ID_PREV")["CNT_INSTALMENT_MATURE_CUM"]
      .diff() == 0
)
rows_without_invariant_fulfiled = credit_card_df[(credit_card_df["CNT_INSTALMENT_IS_CONSTANT_VS_PREV"] == True) & (credit_card_df["AMT_BALANCE"] > 0)]
dtale.show(recreate_and_sort_series_given_rows(rows_without_invariant_fulfiled))
#Seems like the debt is created can have "delay" util start to be paied. But is easy to detect because if we change "AMT_BALANCE" for "AMT_INST_MIN_REGULARITY" > 0 don't exist cases where this condition
#Is true. So the time of the counter of "CNT_INSTALMENT_MATURE_CUM" start to run in the moment where the minimun ammout to don't incurry in delincuency is setted, and stop in the moment "AMT_BALANCE" hits 0.


Casos de analisis: AMT_DRAWINGS_CURRENT distinto de 0 y AMT_DRAWINGS_ATM_CURRENT y compania en null. 
los nulls de Min regularity, no deberia existir deuda a pagar en escenarios donde no hay un valor minimo establecido para no caer en mora.
